# Load Data

In [ ]:
# mini MNIST
# sampled from original MNIST data, e.g., https://huggingface.co/datasets/ylecun/mnist

import pandas as pd
import numpy as np

x = pd.read_csv("mnist-imgs-mini.csv.bz2", index_col=0)
x = x/255.0
x = x.to_numpy()
print(x.shape, x.dtype)

y = pd.read_csv("mnist-lbls-mini.csv.bz2", index_col=0)
y = y.to_numpy()
print(y.shape, y.dtype)

label_names = range(10)
labels = list(label_names)
print(label_names)

n_channels = 1

In [ ]:
# Plot some random examples
import matplotlib as mpl
import matplotlib.pyplot as plt
import math

# based on https://www.kaggle.com/code/gainknowledge/mnist-scikit-learn-tutorial
# Apache 2.0 License

width = int(math.sqrt(x.shape[1] / n_channels))

def displayData(X,Y):
    # set up array
    grid = 7
    fig, ax = plt.subplots(nrows=grid, ncols=grid, figsize=(15,15))
    # loop over randomly drawn numbers
    for i in range(grid):
        for j in range(grid):
            ind = np.random.randint(X.shape[0])
            if (n_channels == 1):
                tmp = X[ind,:].reshape(width, width)
                ax[i,j].imshow(tmp, cmap='gray_r') # display it as gray colors.
            else:
                tmp = X[ind,:].reshape(width, width, n_channels)
                ax[i,j].imshow(tmp) # display it as rgb colors.
            ax[i,j].set_title(label_names[Y[ind][0]])
            ax[i,j].axis('off')
    
    fig.subplots_adjust(hspace=0.5, wspace=0.5)

print(x.shape)
displayData(x,y)

# Choose Random Seed

In [ ]:
# seed numpy for reproducability
from pathlib import Path
import secrets

# delete the file to get a new set of images
seed_path = Path("random-seed.txt")

if seed_path.exists():
    s = int(seed_path.read_text(encoding="utf-8").strip())
    print(f"loaded seed {s} from {seed_path}")
else:
    s = secrets.randbits(32)
    seed_path.write_text(f"{s}\n", encoding="utf-8")
    print(f"generated seed {s} and stored it in {seed_path}")

np.random.seed(s)

# Train Target Classifier

In [ ]:
# train SVM on real classes
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score

# Choose only part of the data set to increase the amount of ambiguity
print(len(x))
n_samples = len(x) // 12
selected_rows = np.random.choice(x.shape[0], size=n_samples)
mask = np.ones(len(x), dtype=bool)
mask[selected_rows] = False
other_rows = np.where(mask)[0]
print(len(selected_rows), len(other_rows))
svm1 = SVC()
true_svm = CalibratedClassifierCV(svm1, ensemble=False)
true_svm.fit(x[selected_rows], np.ravel(y[selected_rows]))
pred = true_svm.predict(x[selected_rows])
#ConfusionMatrixDisplay.from_estimator(true_svm, x[selected_rows], y[selected_rows], labels=labels, normalize="true", values_format='.2f')
ConfusionMatrixDisplay.from_predictions(y[selected_rows], pred, labels=labels, normalize="true", values_format='.2f')
print(label_names)
print(f'f1_train = {f1_score(y[selected_rows], pred, average="micro", labels=labels)}')
plt.show()

pred_test = true_svm.predict(x[other_rows])
ConfusionMatrixDisplay.from_predictions(y[other_rows], pred_test, labels=labels, normalize="true", values_format='.2f')
print(f'f1_test = {f1_score(y[other_rows], pred_test, average="micro", labels=labels)}')
plt.show()

In [ ]:
# save svm for later reproduction
import pickle
with open("cls1.pkl", "wb") as f:
    pickle.dump(true_svm, f, protocol=5)

In [ ]:
# optional: load svm
import pickle
with open("cls1.pkl", "rb") as f:
    true_svm = pickle.load(f)

In [ ]:
# optional: verify accuracy of true SVM, e.g., after loading
pred = true_svm.predict(x[selected_rows])
#ConfusionMatrixDisplay.from_estimator(true_svm, x[selected_rows], y[selected_rows], labels=labels, normalize="true", values_format='.2f')
ConfusionMatrixDisplay.from_predictions(y[selected_rows], pred, labels=labels, normalize="true", values_format='.2f')
print(label_names)
print(f'f1_train = {f1_score(y[selected_rows], pred, average="micro", labels=labels)}')
plt.show()

pred_test = true_svm.predict(x[other_rows])
ConfusionMatrixDisplay.from_predictions(y[other_rows], pred_test, labels=labels, normalize="true", values_format='.2f')
print(f'f1_test = {f1_score(y[other_rows], pred_test, average="micro", labels=labels)}')
plt.show()

# Create Fake Classes

In [ ]:
# create fake classes
from sklearn.preprocessing import label_binarize
from sklearn.feature_selection import r_regression
from scipy.stats import rankdata

print(f'input data shape: {x.shape}')
n_pixels = x.shape[1]
width = int(math.sqrt(n_pixels / n_channels))

# adapt number of sampled combinations to optimize running time and independence
n_random_combinations = 200

lin_comb_weights = np.zeros((n_pixels, n_random_combinations), dtype=np.float32)
idx=list(range(width))
weights=np.ones(width, dtype=np.float32)
for i in range(int(width/2)):
    weights[i] = -1


n_comb = lin_comb_weights.shape[1]
for row in range(n_comb):
    cur_permutation = np.random.permutation(idx)
    cur_weights = np.random.permutation(weights)
    for i in range(width):
        lin_comb_weights[i*width+cur_permutation[i]][row] = cur_weights[i]

#print(lin_comb_weights.shape)

def lin_combin(x):
    return np.dot(x, lin_comb_weights)

values = np.apply_along_axis(lin_combin, 1, x)
# print(values.shape)

# z-transform
means = np.mean(values, 0)
std = np.std(values, 0)
values = np.apply_along_axis(lambda row: row - means, 1, values)
values = np.apply_along_axis(lambda row: row / std, 1, values)

np.set_printoptions(precision=2, suppress=True)

label_dummies = label_binarize(y, classes=labels)
# print(label_dummies.shape)

n_classes = len(labels)

corrs = np.corrcoef(label_dummies, values, rowvar=False)
cross_corrs = corrs[n_classes:,0:n_classes]

max_cross_corrs = np.apply_along_axis(np.max, 1, cross_corrs)

# only keep the combinations with the smallest cross corr with the real labels
while max_cross_corrs.shape[0] > n_classes:
    max_corr_idx = np.argmax(max_cross_corrs)

    max_cross_corrs = np.delete(max_cross_corrs, max_corr_idx, 0)
    values = np.delete(values, max_corr_idx, 1)
    lin_comb_weights = np.delete(lin_comb_weights, max_corr_idx, 1)

# recompute correlations
corrs = np.corrcoef(label_dummies, values, rowvar=False)
cross_corrs = corrs[n_classes:,0:n_classes]

cross_corrs = cross_corrs.flatten()
#print(cross_corrs)
print("cross-correlations:", 
      f'max={np.max(np.abs(cross_corrs))}',
      f'median={np.median(np.abs(cross_corrs))}')

inter_corrs = np.triu(corrs[n_classes:,n_classes:], k=1)
#print(inter_corrs)
n_inter_corrs = int(inter_corrs.shape[0] * (inter_corrs.shape[0]-1) / 2)
flat_inter_corrs = np.empty((n_inter_corrs))
idx = 0
for a in range(inter_corrs.shape[0]):
    for b in range(a+1, inter_corrs.shape[0]):
        flat_inter_corrs[idx] = inter_corrs[a][b]
        idx += 1

print("inter-correlations:", 
      f'max={np.max(np.abs(flat_inter_corrs))}', 
      f'median={np.median(np.abs(flat_inter_corrs))}')

# transform to ranks
values = np.apply_along_axis(lambda col: rankdata(col), 0, values)

fake_classes = np.apply_along_axis(np.argmax, 1, values)
#print(fake_classes.shape)

ConfusionMatrixDisplay.from_predictions(y, fake_classes, labels=labels, normalize="true", values_format='.2f')
plt.show()

# f1 should be close to guessing level, e.g., 1 / n_classes
print(f'f1 = {f1_score(y, fake_classes, average="micro", labels=labels)}')


# Train Bogus Classifier

In [ ]:
# train SVM on fake classes

svm2 = SVC(kernel='linear')
fake_svm = CalibratedClassifierCV(svm2, ensemble=False)
fake_svm.fit(x[selected_rows], fake_classes[selected_rows])
pred_fake = fake_svm.predict(x[selected_rows])

ConfusionMatrixDisplay.from_predictions(fake_classes[selected_rows], pred_fake, labels=labels, normalize="true", values_format='.2f')
plt.show()
# f1_train should be very close to 1
print(f'f1_train = {f1_score(fake_classes[selected_rows], pred_fake, average="micro", labels=labels)}')

ConfusionMatrixDisplay.from_predictions(y[selected_rows], pred_fake, labels=labels, normalize="true", values_format='.2f')
plt.show()
# f1_true should be close to guessing level
print(f'f1_true = {f1_score(y[selected_rows], pred_fake, average="micro", labels=labels)}')


In [ ]:
# save fake svm
import pickle
with open("cls2.pkl", "wb") as f:
    pickle.dump(fake_svm, f, protocol=5)

In [ ]:
# optional: load fake svm
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score

import pickle
with open("cls2.pkl", "rb") as f:
    fake_svm = pickle.load(f)

In [ ]:
# optional: verify independence of fake SVM (e.g., after loading pickle file)
import matplotlib.pyplot as plt

pred_fake = fake_svm.predict(x)

ConfusionMatrixDisplay.from_predictions(y, pred_fake, labels=labels, normalize="true", values_format='.2f')
plt.show()
# f1_true should be close to guessing level
print(f'f1_true = {f1_score(y, pred_fake, average="micro", labels=labels)}')


# Collect Cases Where Target and Bogus Classifier (somewhat) Agree

In [ ]:
# find cases where the classifiers agree (1st and 2nd guess)
import math
from scipy.stats import norm

pred_true_p = true_svm.predict_proba(x[other_rows])
pred_true = true_svm.predict(x[other_rows])
pred_fake_p = fake_svm.predict_proba(x[other_rows])
pred_fake = fake_svm.predict(x[other_rows])
ground_truth = y[other_rows]

n_identical = 0
cls_abab=[]
cls_baba=[]
cls_abba=[]
cls_baab=[]
true_mismatch_proba = 0
fake_mismatch_proba = 0

for i, image_idx in enumerate(other_rows):
    # consistency check for probabilities, this used to be a problem in older
    # versions of sklearn
    pred_true_1st = np.argmax(pred_true_p[i])
    if pred_true_1st != pred_true[i]:
        print(f"true svm mismatch #{image_idx}: pred={pred_true[i]} max_prob={pred_true_1st} p={pred_true_p[i][pred_true_1st]} y={y[i][0]}")
        true_mismatch_proba = true_mismatch_proba + 1
        continue
    pred_fake_1st = np.argmax(pred_fake_p[i])
    if pred_fake_1st != pred_fake[i]:
        print(f"fake svm mismatch #{image_idx}: pred={pred_fake[i]} max_prob={pred_fake_1st} p={pred_fake_p[i][pred_fake_1st]} y={fake_classes[i]}")
        fake_mismatch_proba = fake_mismatch_proba + 1
        continue

    p_t1st = pred_true_p[i][pred_true_1st]
    p_f1st = pred_fake_p[i][pred_fake_1st]
    pred_true_p[i][pred_true_1st] = 0 
    pred_fake_p[i][pred_fake_1st] = 0 

    pred_true_2nd = np.argmax(pred_true_p[i])
    pred_fake_2nd = np.argmax(pred_fake_p[i])
    p_t2nd = pred_true_p[i][pred_true_2nd]
    p_f2nd = pred_fake_p[i][pred_fake_2nd]

    # ignore if true classifier is super shure
    if p_t1st - p_t2nd > .8: continue

    # is there an interesting mixup?
    if (pred_true_1st == pred_fake_1st) and (pred_true_2nd == pred_fake_2nd):
        # 1st and 2nd identical
        n_identical = n_identical + 1
        if pred_true_1st == ground_truth[i]:
            cls_abab.append([image_idx, pred_true_1st, pred_true_2nd, ground_truth[i][0], [p_t1st, p_t2nd], [p_f1st, p_f2nd], 'abab'])
        elif pred_true_2nd == ground_truth[i]:
            cls_baba.append([image_idx, pred_true_1st, pred_true_2nd, ground_truth[i][0], [p_t1st, p_t2nd], [p_f1st, p_f2nd], 'baba'])
    elif (pred_true_1st == pred_fake_2nd) and (pred_true_2nd == pred_fake_1st):
        # 1st and 2nd mixed up
        if pred_true_1st == ground_truth[i]:
            cls_abba.append([image_idx, pred_true_1st, pred_true_2nd, ground_truth[i][0], [p_t1st, p_t2nd], [p_f1st, p_f2nd], 'abba'])
        elif pred_true_2nd == ground_truth[i]:
            cls_baab.append([image_idx, pred_true_1st, pred_true_2nd, ground_truth[i][0], [p_t1st, p_t2nd], [p_f1st, p_f2nd], 'baab'])

# probabilities may not completely match the prediction, see sklearn documentation
print(f"mismatch between pred and probs: true={true_mismatch_proba/n_samples} fake={fake_mismatch_proba/n_samples}")

print(f"complete matches {(len(cls_abab) + len(cls_baba)) / n_samples}")
print(f"abab {len(cls_abab)}, baba {len(cls_baba)}, abba {len(cls_abba)}, baab {len(cls_baab)}")

# Create LIME Visualizations

In [ ]:
# get centroids for correlation
centroids_df = pd.read_csv("centroids.csv.bz2", index_col=0)
print(centroids_df.shape)

print(centroids_df.T.corr())

In [ ]:
# explain with lime
# cmp https://github.com/marcotcr/lime/blob/master/doc/notebooks/Tutorial%20-%20MNIST%20and%20RF.ipynb

from lime.lime_image import LimeImageExplainer
from lime.wrappers.scikit_image import SegmentationAlgorithm
from skimage import color as sk_color
from skimage.color import label2rgb
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt

n_pixels = x.shape[1]
width = int(math.sqrt(n_pixels / n_channels))

setup = [[1, true_svm], [2, fake_svm]]

explainer = LimeImageExplainer()
# segmenter -> return int matrix of same shape with superpixel index in each cell

def split_evenly(img, n_splits):
    grid = np.zeros((img.shape[0], img.shape[1]), dtype=int)
    grid_width_x = img.shape[0] / n_splits
    grid_width_y = img.shape[1] / n_splits
    for col in range(img.shape[0]):
        x_grid = math.trunc(col / grid_width_x)
        for row in range(img.shape[1]):
            y_grid = math.trunc(row / grid_width_y)
            grid[col][row] = x_grid + y_grid * n_splits
    return grid


grids = {}
def grid_n_parts(img, splits):
    grid_idx = (img.shape[0], img.shape[1], splits)
    if grid_idx in grids:
        return grids[grid_idx]
    else:
        grid = split_evenly(img, splits)
        grids[grid_idx] = grid
        return grid

def grid_5parts(img):
    return grid_n_parts(img, 5)

def grid_7parts(img):
    return grid_n_parts(img, 7)

def grid_11parts(img):
    return grid_n_parts(img, 11)

# 'quickshift', 'slic', 'felzenszwalb'
# https://scikit-image.org/docs/stable/api/skimage.segmentation.html#skimage.segmentation.quickshift
# ratio: color=1 space=0
quickshift = SegmentationAlgorithm('quickshift')
# https://scikit-image.org/docs/stable/api/skimage.segmentation.html#skimage.segmentation.felzenszwalb
felzenszwalb = SegmentationAlgorithm('felzenszwalb', sigma=width/20, scale=width/50, channel_axis=2)
# https://scikit-image.org/docs/stable/api/skimage.segmentation.html#skimage.segmentation.slic
slic_sma = SegmentationAlgorithm('slic', n_segments=25, compactness=.01, convert2lab=False, enforce_connectivity=True)
slic_mid = SegmentationAlgorithm('slic', n_segments=25, compactness=0.1, convert2lab=False, enforce_connectivity=True)
slic_lrg = SegmentationAlgorithm('slic', n_segments=25, compactness=1.0, convert2lab=False, enforce_connectivity=True)


## in case different aggregation functions should be used for some of the channels (e.g. value)
def average_hsvs(imgs):
    c = np.zeros_like(imgs[0])
    for img in imgs:
        c += img
    c /= len(imgs)
    return c

## combine several hsv images, with hue being weighted by the saturation
def combine_hsvs(imgs):
    c = np.zeros_like(imgs[0])
    for col in range(c.shape[0]):
        for row in range(c.shape[1]):
            huevec = [0, 0]
            saturation = 0
            value = 0
            for img in imgs:
                huevec[0] += math.cos(img[col][row][0]*math.pi*2) * img[col][row][1]
                huevec[1] += math.sin(img[col][row][0]*math.pi*2) * img[col][row][1]
                saturation += img[col][row][1]
                value += img[col][row][2]

            c[col][row][0] = math.atan2(huevec[1], huevec[0])/math.pi/2
            c[col][row][1] = saturation / len(imgs)
            c[col][row][2] = value / len(imgs)
    return c

def combine_weights(ws):
    c = np.zeros_like(ws[0])
    for w in ws:
        c += w
    norm = max(abs(np.min(c)), np.max(c))
    c /= norm
    return c

white = np.ones((width,width))
white_rgb = np.ones((width,width,3))
HUE_RED = 0.0 # 0 deg
HUE_GREEN = 1.0/3.0 # 120 deg
VALUE_ADD = 0.5 # how much the computed weight also increases the value

# choose how many examples per combination ab/ab, ab/ba, ba/ab, ba/ba
N_SAMPLES = 2

case_map = []
faithfulness = []

# use numpy here to get a result depending on the seed chosen above
def sample_trials(trials, n):
    n = min(n, len(trials))
    selected_idx = np.random.choice(len(trials), size=n, replace=False)
    return [trials[i] for i in selected_idx]

for trial in sample_trials(cls_baab, N_SAMPLES) + sample_trials(cls_baba, N_SAMPLES) + sample_trials(cls_abba, N_SAMPLES) + sample_trials(cls_abab, N_SAMPLES):
    idx = trial[0]
    pred_1st = trial[1]
    pred_2nd = trial[2]
    true_class = trial[3]
    case_type = trial[6]

    print(f'{idx}: 1st {pred_1st}, 2nd {pred_2nd}, true {true_class}, c_1 {trial[4]}, c_2 {trial[5]}, {case_type}')
    faith = {'idx': idx, 'trial_type': case_type, 'ground': true_class}

    for s in setup:
        clsf_id = s[0]
        classifier = s[1]
        other_classifier = setup[2-clsf_id][1]

        def predict_instance(img_batch):
            if n_channels == 1:
                # back to gray scale and adjust dimensionality
                gray = sk_color.rgb2gray(img_batch).reshape((-1, width*width))
                pred = classifier.predict_proba(gray)
            else:
                # adjust dimensionality
                pred = classifier.predict_proba(img_batch.reshape((-1, width*width*n_channels)))

            return pred

        if n_channels == 1:
            img_data = x[idx].reshape((width, width))
            img = sk_color.gray2rgb(img_data)
            # MNIST is inverted
            orig_img = sk_color.gray2rgb(white - img_data)
        else:
            img_data = x[idx].reshape((width, width, n_channels))
            # no need to invert
            img = img_data
            orig_img = img


        def run_lime(segm_fn):
            explanation = explainer.explain_instance(img, predict_instance, 
                                                    num_features=100,
                                                    labels=(pred_1st, pred_2nd),
                                                    hide_color=None, num_samples=1000,
                                                    segmentation_fn=segm_fn,
                                                    batch_size=100)

            p1st_local_weights = explanation.local_exp[pred_1st] # list of (feat-id, weight)
            p2nd_local_weights = explanation.local_exp[pred_2nd]

            p1st_max_weight = abs(p1st_local_weights[0][1])
            p2nd_max_weight = abs(p2nd_local_weights[0][1])

            p1weights = {}
            for w in p1st_local_weights:
                p1weights[w[0]] = w[1]
            p2weights = {}
            for w in p2nd_local_weights:
                p2weights[w[0]] = w[1]

            super_pixels = explanation.segments
            w1 = np.zeros_like(super_pixels, dtype=np.float32)
            w2 = np.zeros_like(super_pixels, dtype=np.float32)

            for col in range(img.shape[0]):
                for row in range(img.shape[1]):
                    pix_id = super_pixels[col][row]
                    if pix_id in p1weights:
                        w1[col][row] = p1weights[pix_id] / p1st_max_weight
                    if pix_id in p2weights:
                        w2[col][row] = p2weights[pix_id] / p2nd_max_weight

            return w1, w2
        

        def weights2mask(w):
            base_color = np.mean(orig_img)
            bg_color = np.array([base_color,base_color,base_color])
            white = np.array([1,1,1])

            result_img = np.copy(orig_img) # todo create empty image in img.size and init with bgcolor?
            for col in range(result_img.shape[0]):
                for row in range(result_img.shape[1]):
                    if w[col][row]>0:
                        weight = math.pow(w[col][row], 0.5)
                        result_img[col][row] = weight * orig_img[col][row] + (1-weight) * white
                    else:
                        result_img[col][row] = white
            return result_img


        def weights2hsv(w):
            result_img = sk_color.rgb2hsv(orig_img)
            for col in range(w.shape[0]):
                for row in range(w.shape[1]):
                    #weight = weight_trans(abs(w[col][row]))
                    weight = math.pow(abs(w[col][row]), 1.25)
                    # saturation
                    result_img[col][row][1] = weight
                    # increase value (to get visible colors in dark areas)
                    result_img[col][row][2] = min(1, weight*VALUE_ADD + result_img[col][row][2])
                    # TODO blend between weight color and original color
                    if w[col][row] > 0:
                        result_img[col][row][0] = HUE_GREEN # positive weight -> green
                    elif w[col][row] < 0: 
                        result_img[col][row][0] = HUE_RED # negative weight -> red
            return result_img
        

        def weights2rgbmap(w):
            result_img = np.copy(orig_img)
            for col in range(w.shape[0]):
                for row in range(w.shape[1]):
                    #weight = weight_trans(abs(w[col][row]))
                    weight = math.pow(abs(w[col][row]), 1.25)
                    # blend between weight color and original color
                    if w[col][row] > 0:
                        # make it green
                        result_img[col][row][0] = result_img[col][row][0] * (1-weight) 
                        result_img[col][row][1] = result_img[col][row][1] * (1-weight) + weight
                        result_img[col][row][2] = result_img[col][row][2] * (1-weight) 
                    elif w[col][row] < 0: 
                        # make it red
                        result_img[col][row][0] = result_img[col][row][0] * (1-weight) + weight
                        result_img[col][row][1] = result_img[col][row][1] * (1-weight)
                        result_img[col][row][2] = result_img[col][row][2] * (1-weight) 
            return result_img

        def make_plot(wimg1, wimg2, with_p, seg_id):
            inverse_order = (clsf_id == 2) and (case_type == 'abba' or case_type == 'baab')
            p1 = math.trunc(100 * trial[3+clsf_id][0] + .5)
            p2 = math.trunc(100 * trial[3+clsf_id][1] + .5)

            fig, ((orig, img_1st, img_2nd)) = plt.subplots(1, 3, figsize=(4,2))
            orig.set_title('Original')
            orig.imshow(orig_img)
            orig.axis('off')

            if with_p:
                img_1st.set_title(f'{p1}% {label_names[pred_2nd if inverse_order else pred_1st]}')
            else:
                img_1st.set_title(f'A: {label_names[pred_2nd if inverse_order else pred_1st]}')
            img_1st.imshow(wimg2 if inverse_order else wimg1)
            img_1st.axis('off')

            if with_p:
                img_2nd.set_title(f'{p2}% {label_names[pred_1st if inverse_order else pred_2nd]}')
            else:
                img_2nd.set_title(f'B: {label_names[pred_1st if inverse_order else pred_2nd]}')
            img_2nd.imshow(wimg1 if inverse_order else wimg2)
            img_2nd.axis('off')
            p_on = 1 if with_p else 0
            plt_name = f"lime-imgs/expl-i{idx}-t{true_class}-c{clsf_id}-{case_type}-{seg_id}-p{p_on}.png"
            plt.savefig(plt_name)
            case_def = {
                'idx': idx, 
                'trial_type': case_type,
                'vis_type': seg_id,
                'with_pct': p_on,
                'trial_target': true_class,
                'clsf_id': clsf_id,
                'file_name': plt_name
            }
            case_map.append(case_def)
            #plt.close()

        try:
            (w1a, w2a) = run_lime(slic_sma)
            (w1b, w2b) = run_lime(slic_mid)
            (w1c, w2c) = run_lime(slic_lrg)
            (w1d, w2d) = run_lime(grid_5parts)
            (w1e, w2e) = run_lime(grid_7parts)
            (w1f, w2f) = run_lime(grid_11parts)
            #(w1f, w2f) = get_expl_weights(felzenszwalb)
            #(w1g, w2g) = get_expl_weights(quickshift)
            w1ac = combine_weights([w1a, w1b, w1c])
            w2ac = combine_weights([w2a, w2b, w2c])
            w1df = combine_weights([w1d, w1e, w1f])
            w2df = combine_weights([w2d, w2e, w2f])
            w1af = combine_weights([w1a, w1b, w1c, w1d, w1e, w1f])
            w2af = combine_weights([w2a, w2b, w2c, w2d, w2e, w2f])

            make_plot(sk_color.hsv2rgb(weights2hsv(w1af)), 
                      sk_color.hsv2rgb(weights2hsv(w2af)), False, "haf")
            if False:
                make_plot(sk_color.hsv2rgb(weights2hsv(w1ac)), 
                        sk_color.hsv2rgb(weights2hsv(w2ac)), True, "hac")
                make_plot(sk_color.hsv2rgb(weights2hsv(w1df)), 
                        sk_color.hsv2rgb(weights2hsv(w2df)), True, "hdf")
                make_plot(sk_color.hsv2rgb(weights2hsv(w1a)), 
                        sk_color.hsv2rgb(weights2hsv(w2a)), False, "hsls")
                make_plot(sk_color.hsv2rgb(weights2hsv(w1b)), 
                        sk_color.hsv2rgb(weights2hsv(w2b)), False, "hslm")
                make_plot(sk_color.hsv2rgb(weights2hsv(w1c)), 
                        sk_color.hsv2rgb(weights2hsv(w2c)), False, "hsll")
                make_plot(sk_color.hsv2rgb(weights2hsv(w1d)), 
                        sk_color.hsv2rgb(weights2hsv(w2d)), False, "hgr7")
                make_plot(sk_color.hsv2rgb(weights2hsv(w1e)), 
                        sk_color.hsv2rgb(weights2hsv(w2e)), False, "hgr11")
                make_plot(sk_color.hsv2rgb(weights2hsv(w1f)), 
                        sk_color.hsv2rgb(weights2hsv(w2f)), True, "hflz")
                make_plot(sk_color.hsv2rgb(weights2hsv(w1g)), 
                        sk_color.hsv2rgb(weights2hsv(w2g)), True, "hqsh")

            # keep masked version for faithfulness estimation
            mask1 = weights2mask(w1af)
            mask2 = weights2mask(w2af)

            make_plot(mask1, mask2, False, "maf")
            if False:
                make_plot(weights2mask(w1ac), 
                        weights2mask(w2ac), True, "mac")
                make_plot(weights2mask(w1df), 
                        weights2mask(w2df), True, "mdf")
                make_plot(weights2mask(w1a), 
                        weights2mask(w2a), False, "msls")
                make_plot(weights2mask(w1b), 
                        weights2mask(w2b), False, "mslm")
                make_plot(weights2mask(w1c), 
                        weights2mask(w2c), False, "msll")
                make_plot(weights2mask(w1d), 
                        weights2mask(w2d), False, "mgr7")
                make_plot(weights2mask(w1e), 
                        weights2mask(w2e), False, "mgr11")
                make_plot(weights2mask(w1f), 
                        weights2mask(w2f), True, "mflz")
                make_plot(weights2mask(w1g), 
                        weights2mask(w2g), True, "mqsh")

            if False:
                make_plot(weights2rgbmap(w1af), 
                        weights2rgbmap(w2af), False, "raf")

            # compute predictions for masked images to estimate faithfulness
            # currently inverts the image, undo for RGB input data
            prob1st = predict_instance(white_rgb - mask1).squeeze()
            prob2nd = predict_instance(white_rgb - mask2).squeeze()
            p1st = np.argmax(prob1st)
            p2nd = np.argmax(prob2nd)
            print(f'#{idx} {case_type}: m1 {p1st}, m2 {p2nd}, p1 {pred_1st}, p2 {pred_2nd}, true {true_class}')
            faith[f'c{clsf_id}_a'] = pred_1st
            faith[f'c{clsf_id}_a_p'] = trial[3+clsf_id][0]
            faith[f'c{clsf_id}_b'] = pred_2nd
            faith[f'c{clsf_id}_b_p'] = trial[3+clsf_id][1]
            faith[f'c{clsf_id}_fa'] = p1st
            faith[f'c{clsf_id}_fa_p'] = prob1st[p1st]
            faith[f'c{clsf_id}_fa_1stp'] = prob1st[pred_1st]
            faith[f'c{clsf_id}_fa_2ndp'] = prob1st[pred_2nd]
            faith[f'c{clsf_id}_fb'] = p2nd
            faith[f'c{clsf_id}_fb_p'] = prob2nd[p2nd]
            faith[f'c{clsf_id}_fb_1stp'] = prob2nd[pred_1st]
            faith[f'c{clsf_id}_fb_2ndp'] = prob2nd[pred_2nd]

            # cross-examination
            mask_as_row = sk_color.rgb2gray(white_rgb - mask1).reshape((-1, width*width))
            prob1st_other = other_classifier.predict_proba(mask_as_row).squeeze()
            faith[f'c{clsf_id}_fa_other_1stp'] = prob1st_other[pred_1st]
            faith[f'c{clsf_id}_fa_other_2ndp'] = prob1st_other[pred_2nd]

            prob2nd_other = other_classifier.predict_proba(
                sk_color.rgb2gray(white_rgb - mask2)
                    .reshape((-1, width*width))).squeeze()
            faith[f'c{clsf_id}_fb_other_1stp'] = prob2nd_other[pred_1st]
            faith[f'c{clsf_id}_fb_other_2ndp'] = prob2nd_other[pred_2nd]

            # pixel-level correlations
            mask_cluster_corr = np.corrcoef(mask_as_row, centroids_df.iloc[true_class].values)[0, 1]
            print(f"corr_mask={mask_cluster_corr}")
            faith[f'c{clsf_id}_corr'] = mask_cluster_corr
            
        except Exception as err:
            print(f"could not create explanation for trial {idx}, classifier {clsf_id}", err)

    faithfulness.append(faith)

pd.DataFrame(case_map).to_csv("cases.csv")
pd.DataFrame(faithfulness).to_csv("faithfulness.csv")


# Convert Case Map to PHP Array

In [ ]:
cases = pd.read_csv("cases.csv", index_col=0)
# print(cases)

# how many items per trial_type should be training trials?
n_train_items = 1

# clsf_id is a within-subject variable, vis_type and with_pct are between-subjects.
# Create one wide table per between-subject condition, with one row per idx and one
# column per classifier containing the corresponding image filename.
condition_tables = {}

for (vis_type, with_pct), group in cases.groupby(["vis_type", "with_pct"], sort=True):
    wide = (
        group.pivot(index=["idx", "trial_type", "trial_target"], columns="clsf_id", values="file_name")
        .rename(columns=lambda clsf_id: f"clsf_{int(clsf_id)}")
        .reset_index()
        .sort_values(["trial_type", "idx"])
    )
    wide["vis_type"] = vis_type
    wide["with_pct"] = int(with_pct)
    wide = wide[["idx", "trial_type", "trial_target", "vis_type", "with_pct", "clsf_1", "clsf_2"]]
    condition_tables[(vis_type, int(with_pct))] = wide

def split_training_trials(df, n_train_items):
    training_parts = []
    trial_parts = []

    for _, group in df.groupby("trial_type", sort=True):
        ordered = group.sort_values("idx")
        split_at = min(n_train_items, len(ordered))
        training_parts.append(ordered.iloc[:split_at])
        trial_parts.append(ordered.iloc[split_at:])

    training_df = pd.concat(training_parts, ignore_index=True).sort_values(["trial_type", "idx"])
    trial_df = pd.concat(trial_parts, ignore_index=True).sort_values(["trial_type", "idx"])
    return training_df.reset_index(drop=True), trial_df.reset_index(drop=True)

def php_scalar(value):
    if pd.isna(value):
        return "null"
    if isinstance(value, (np.integer, int)):
        return str(int(value))
    text = str(value).replace("\\", "\\\\").replace("'", "\\'")
    return f"'{text}'"

def dataframe_to_php_array(df):
    rows = []
    for row in df.to_dict(orient="records"):
        entries = [f"'{key}' => {php_scalar(value)}" for key, value in row.items()]
        rows.append("    [" + ", ".join(entries) + "]")
    return "[\n" + ",\n".join(rows) + "\n]"

php_arrays = []
for (vis_type, with_pct), table in condition_tables.items():
    training_table, trial_table = split_training_trials(table, n_train_items)

    print()
    print(f"condition={(vis_type, with_pct)}")
    print("training")
    print(training_table)
    print("trial")
    print(trial_table)

    base_name = f"cases_{vis_type}_p{with_pct}"
    php_arrays.append(
        f"$training_{base_name} = {dataframe_to_php_array(training_table)};"
    )
    php_arrays.append(
        f"$trial_{base_name} = {dataframe_to_php_array(trial_table)};"
    )

php_code = "<?php\n\n" + "\n\n".join(php_arrays) + "\n"
with open("cases.php", "w", encoding="utf-8") as f:
    f.write(php_code)

print()
print(php_code)
print()
print("Saved PHP arrays to cases.php")